In [ ]:
#!/usr/bin/env python
import os
import glob
import numpy as np
import pandas as pd

from statsforecast import StatsForecast
from statsforecast.models import AutoETS

# ------------------------------------------------------------------
# CONFIG (stable84)
# ------------------------------------------------------------------
RUN_TAG = "251110"

PREFIX_INPUT_FOLDER = rf"out/{RUN_TAG}/prefix_datasets_stable84"
OUTPUT_FOLDER       = rf"out/{RUN_TAG}/prefix_with_ets_stable84"

# Verenich-style prefixes may or may not have a meaningful variant column.
# Keep this None to disable per-variant ETS by default.
# Or set e.g. "process" if you want per-variant models.
VARIANT_COLUMN = None  # e.g. "process"

AUTOETS_MODEL      = "ZZN"
AUTOETS_SEASON_LEN = 1
# ------------------------------------------------------------------


def load_prefix_file(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    required = [
        "case_id",
        "timestamp",
        "elapsed_time",
        "remaining_time",
        "case_start_time",
        "case_complete_time",
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{path} missing required column(s): {missing}")

    return df


def build_case_table(df_prefix: pd.DataFrame, variant_col: str | None = None) -> pd.DataFrame:
    base_cols = ["case_id", "case_start_time", "case_complete_time"]
    cols = base_cols.copy()

    if variant_col is not None and variant_col in df_prefix.columns:
        cols.append(variant_col)

    cases = (
        df_prefix
        .sort_values(["case_id", "timestamp"])
        .drop_duplicates(subset=["case_id"], keep="first")[cols]
        .copy()
    )
    cases["cycle_time"] = cases["case_complete_time"] - cases["case_start_time"]
    cases = cases.sort_values("case_start_time").reset_index(drop=True)
    return cases


def build_ts_df(cases: pd.DataFrame, unique_id: str) -> pd.DataFrame:
    n = len(cases)
    return pd.DataFrame({
        "unique_id": unique_id,
        "ds": np.arange(n),
        "y": cases["cycle_time"].to_numpy(),
    })


def fit_autoets(train_ts: pd.DataFrame) -> StatsForecast:
    sf = StatsForecast(
        models=[AutoETS(model=AUTOETS_MODEL, season_length=AUTOETS_SEASON_LEN)],
        freq=1,
    )
    sf.fit(df=train_ts)
    return sf


def forecast_cycle_times(sf: StatsForecast, h: int) -> np.ndarray:
    y_hat = sf.predict(h=h)
    preds = y_hat["AutoETS"].to_numpy()
    if len(preds) != h:
        raise RuntimeError(f"Expected {h} forecasts, got {len(preds)}")
    return preds


def add_ets_predictions_to_prefix(
    df_prefix: pd.DataFrame,
    D_hat: np.ndarray,
    case_ids_in_order: np.ndarray,
    col_D: str,
    col_R: str,
) -> pd.DataFrame:
    df_prefix = df_prefix.copy()

    case_pred_df = pd.DataFrame({"case_id": case_ids_in_order, col_D: D_hat})
    df_prefix = df_prefix.merge(case_pred_df, on="case_id", how="left")

    df_prefix[col_R] = np.clip(
        df_prefix[col_D] - df_prefix["elapsed_time"],
        a_min=0.0,
        a_max=None,
    )
    return df_prefix


def fit_and_predict_per_variant(base_root, cases_train, cases_val, cases_test, variant_col: str):
    if variant_col not in cases_train.columns:
        print(f"  [per-variant] Column '{variant_col}' not in train -> skip.")
        return {}, {}

    variants = sorted(cases_train[variant_col].dropna().unique())
    if not variants:
        print("  [per-variant] No variants in train -> skip.")
        return {}, {}

    val_pred_by_case, test_pred_by_case = {}, {}

    for v in variants:
        train_v = cases_train[cases_train[variant_col] == v].copy()
        val_v   = cases_val[cases_val[variant_col] == v].copy() if variant_col in cases_val.columns else cases_val.iloc[0:0].copy()
        test_v  = cases_test[cases_test[variant_col] == v].copy() if variant_col in cases_test.columns else cases_test.iloc[0:0].copy()

        if len(train_v) == 0:
            continue

        unique_id = f"{base_root}_var_{v}"
        ts_train_v = build_ts_df(train_v, unique_id=unique_id)
        sf_v = fit_autoets(ts_train_v)

        h_val_v = len(val_v)
        h_test_v = len(test_v)
        h_total_v = h_val_v + h_test_v
        if h_total_v == 0:
            continue

        D_hat_all_v = forecast_cycle_times(sf_v, h=h_total_v)
        D_hat_val_v = D_hat_all_v[:h_val_v]
        D_hat_test_v = D_hat_all_v[h_val_v:]

        for cid, d in zip(val_v["case_id"].to_numpy(), D_hat_val_v):
            val_pred_by_case[cid] = d
        for cid, d in zip(test_v["case_id"].to_numpy(), D_hat_test_v):
            test_pred_by_case[cid] = d

    return val_pred_by_case, test_pred_by_case


def process_scenario(train_path: str):
    base_name = os.path.basename(train_path)
    base_root = base_name.replace("_train_prefix.csv", "")
    print(f"\n=== Scenario: {base_root} ===")

    val_path  = os.path.join(PREFIX_INPUT_FOLDER, f"{base_root}_val_prefix.csv")
    test_path = os.path.join(PREFIX_INPUT_FOLDER, f"{base_root}_test_prefix.csv")

    if not os.path.exists(val_path) or not os.path.exists(test_path):
        print("  Skipping: missing val or test")
        return

    df_train = load_prefix_file(train_path)
    df_val   = load_prefix_file(val_path)
    df_test  = load_prefix_file(test_path)

    print(f"  Train rows: {len(df_train)}, cases: {df_train['case_id'].nunique()}")
    print(f"  Val   rows: {len(df_val)},   cases: {df_val['case_id'].nunique()}")
    print(f"  Test  rows: {len(df_test)},  cases: {df_test['case_id'].nunique()}")

    cases_train = build_case_table(df_train, variant_col=VARIANT_COLUMN)
    cases_val   = build_case_table(df_val,   variant_col=VARIANT_COLUMN)
    cases_test  = build_case_table(df_test,  variant_col=VARIANT_COLUMN)

    if len(cases_train) == 0:
        print("  No train cases -> skip.")
        return

    ts_train = build_ts_df(cases_train, unique_id=base_root)
    sf_global = fit_autoets(ts_train)

    h_val  = len(cases_val)
    h_test = len(cases_test)
    h_total = h_val + h_test
    if h_total == 0:
        print("  No val/test cases -> skip.")
        return

    D_hat_all = forecast_cycle_times(sf_global, h=h_total)
    D_hat_val = D_hat_all[:h_val]
    D_hat_test = D_hat_all[h_val:]

    df_val_ets = add_ets_predictions_to_prefix(
        df_val, D_hat_val, cases_val["case_id"].to_numpy(),
        col_D="D_hat_ets", col_R="R_hat_ets"
    )
    df_test_ets = add_ets_predictions_to_prefix(
        df_test, D_hat_test, cases_test["case_id"].to_numpy(),
        col_D="D_hat_ets", col_R="R_hat_ets"
    )

    # Optional per-variant
    if VARIANT_COLUMN is not None:
        val_pred_by_case, test_pred_by_case = fit_and_predict_per_variant(
            base_root, cases_train, cases_val, cases_test, variant_col=VARIANT_COLUMN
        )

        if val_pred_by_case:
            df_val_ets = df_val_ets.merge(
                pd.DataFrame({"case_id": list(val_pred_by_case.keys()),
                              "D_hat_ets_var": list(val_pred_by_case.values())}),
                on="case_id", how="left"
            )
            df_val_ets["R_hat_ets_var"] = np.clip(
                df_val_ets["D_hat_ets_var"] - df_val_ets["elapsed_time"], 0.0, None
            )

        if test_pred_by_case:
            df_test_ets = df_test_ets.merge(
                pd.DataFrame({"case_id": list(test_pred_by_case.keys()),
                              "D_hat_ets_var": list(test_pred_by_case.values())}),
                on="case_id", how="left"
            )
            df_test_ets["R_hat_ets_var"] = np.clip(
                df_test_ets["D_hat_ets_var"] - df_test_ets["elapsed_time"], 0.0, None
            )

    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    out_val  = os.path.join(OUTPUT_FOLDER, f"{base_root}_val_prefix_ets.csv")
    out_test = os.path.join(OUTPUT_FOLDER, f"{base_root}_test_prefix_ets.csv")

    df_val_ets.to_csv(out_val, index=False)
    df_test_ets.to_csv(out_test, index=False)

    print(f"  Saved:\n    {out_val}\n    {out_test}")


def main():
    if not os.path.isdir(PREFIX_INPUT_FOLDER):
        raise SystemExit(f"Prefix folder not found: {PREFIX_INPUT_FOLDER}")

    train_files = sorted(glob.glob(os.path.join(PREFIX_INPUT_FOLDER, "*_train_prefix.csv")))
    if not train_files:
        raise SystemExit(f"No '*_train_prefix.csv' files in {PREFIX_INPUT_FOLDER}")

    print(f"Found {len(train_files)} scenario(s) to process.")
    for train_path in train_files:
        process_scenario(train_path)


if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python
import os
import glob
import numpy as np
import pandas as pd

# ------------------------------------------------------------------
RUN_TAG = "251110"
ETS_FOLDER  = rf"out/{RUN_TAG}/prefix_with_ets_stable84"
VAL_PATTERN = "*_val_prefix_ets.csv"
# ------------------------------------------------------------------


def _mae_mape(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[float, float]:
    mae = float(np.mean(np.abs(y_true - y_pred)))

    mask = y_true != 0
    if mask.any():
        mape = float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100.0)
    else:
        mape = np.nan

    return mae, mape


def compute_metrics_for_file(path: str):
    df = pd.read_csv(path)

    if "remaining_time" not in df.columns:
        raise ValueError(f"{path} missing required column: remaining_time")
    if "R_hat_ets" not in df.columns:
        raise ValueError(f"{path} missing required column: R_hat_ets")

    y_true = df["remaining_time"].to_numpy()
    y_pred_global = df["R_hat_ets"].to_numpy()
    mae_g, mape_g = _mae_mape(y_true, y_pred_global)

    if "R_hat_ets_var" in df.columns:
        y_pred_var = df["R_hat_ets_var"].to_numpy()
        mae_v, mape_v = _mae_mape(y_true, y_pred_var)
    else:
        mae_v, mape_v = np.nan, np.nan

    return mae_g, mape_g, mae_v, mape_v


def main():
    if not os.path.isdir(ETS_FOLDER):
        raise SystemExit(f"ETS folder not found: {ETS_FOLDER}")

    val_files = sorted(glob.glob(os.path.join(ETS_FOLDER, VAL_PATTERN)))
    if not val_files:
        raise SystemExit(f"No '{VAL_PATTERN}' files found in {ETS_FOLDER}")

    print(f"Found {len(val_files)} validation file(s).")

    maes_g, mapes_g = [], []
    maes_v, mapes_v = [], []

    for path in val_files:
        mae_g, mape_g, mae_v, mape_v = compute_metrics_for_file(path)
        maes_g.append(mae_g); mapes_g.append(mape_g)
        maes_v.append(mae_v); mapes_v.append(mape_v)

        print(
            f"{os.path.basename(path)} -> "
            f"GLOBAL: MAE={mae_g:.4f}, MAPE={mape_g:.2f}% | "
            f"PER-VARIANT: MAE={mae_v:.4f}, MAPE={mape_v:.2f}%"
        )

    print("\n------------------------------")
    print("GLOBAL ETS over all val files:")
    print(f"  Average MAE  : {float(np.mean(maes_g)):.4f}")
    print(f"  Average MAPE : {float(np.nanmean(mapes_g)):.2f}%")

    print("\nPER-VARIANT ETS over all val files (ignoring NaNs):")
    print(f"  Average MAE  : {float(np.nanmean(maes_v)):.4f}")
    print(f"  Average MAPE : {float(np.nanmean(mapes_v)):.2f}%")


if __name__ == "__main__":
    main()
